## Data Modeling
This notebook includes the data modeling steps in the solar energy project. <br>
As inputs it requires the cleaned, encoded, re-sampled, and scaled data frames X_train, X_test, y_train, y_test. <br>
The objective is to test different models

In [3]:
# all in one code
import pandas as pd
path = r"..\data\processed\\" # for Manuel

# Data collection

# Import os_minmax
X_train = pd.read_csv(path+'X_train_os_minmax.csv')
X_test = pd.read_csv(path+'X_test_minmax.csv')
y_train = pd.read_csv(path+'y_train_os.csv').squeeze()
y_test = pd.read_csv(path+'y_test.csv').squeeze()

# # Import os_robust
# X_train = pd.read_csv(path+'X_train_os_robust.csv')
# X_test = pd.read_csv(path+'X_test_robust.csv')
# y_train = pd.read_csv(path+'y_train_os.csv').squeeze()
# y_test = pd.read_csv(path+'y_test.csv').squeeze()

# # Import us_robust
# X_train = pd.read_csv(path+'X_train_us_robust.csv')
# X_test = pd.read_csv(path+'X_test_robust.csv')
# y_train = pd.read_csv(path+'y_train_us.csv').squeeze()
# y_test = pd.read_csv(path+'y_test.csv').squeeze()

# Grid Search
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, make_scorer
param_grids = {
    "RandomForest": {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5, 10]
    },
    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale", "auto"]
    },
    "XGBoost": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 0.2],
        "max_depth": [3, 5, 7]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    }
}
models = {
    "RandomForest": RandomForestClassifier(random_state=42),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "KNN": KNeighborsClassifier()
}
scorer = make_scorer(f1_score, average='binary')
best_models = {}
for name, model in models.items():
    print(f" Tuning {name} for F1-score...")
    grid_search = GridSearchCV(model, param_grids[name], cv=5, scoring=scorer, n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    best_models[name] = grid_search.best_estimator_
    print(f" Best {name} Params: {grid_search.best_params_}")
    print(f" Best {name} F1-score: {grid_search.best_score_:.4f}\n")

# Evaluate the best models on the test set
print("Final Model Evaluations on Test Data:")
for name, model in best_models.items():
    y_train_pred = model.predict(X_train)
    f1 = f1_score(y_train, y_train_pred)
    print(f" {name} train: F1-score = {f1:.4f}")
    y_test_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_test_pred)
    print(f" {name} test: F1-score = {f1:.4f}")

 Tuning RandomForest for F1-score...
 Best RandomForest Params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
 Best RandomForest F1-score: 0.9580

 Tuning SVM for F1-score...
 Best SVM Params: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
 Best SVM F1-score: 0.8904

 Tuning XGBoost for F1-score...


c:\Users\manym\Documents\Data Scientest project\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:08:00] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


 Best XGBoost Params: {'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200}
 Best XGBoost F1-score: 0.9543

 Tuning KNN for F1-score...
 Best KNN Params: {'metric': 'euclidean', 'n_neighbors': 9, 'weights': 'distance'}
 Best KNN F1-score: 0.9438

Final Model Evaluations on Test Data:
 RandomForest train: F1-score = 0.9982
 RandomForest test: F1-score = 0.9497
 SVM train: F1-score = 0.8935
 SVM test: F1-score = 0.9238
 XGBoost train: F1-score = 0.9998
 XGBoost test: F1-score = 0.9464
 KNN train: F1-score = 1.0000
 KNN test: F1-score = 0.9375


### 0. Data collection

In [5]:
import pandas as pd
path = r"..\data\processed\\" # for Manuel

In [3]:
path = r"..\data\processed\\" # for Manuel

X_train = pd.read_csv(path+'X_train_os_minmax.csv')
X_test = pd.read_csv(path+'X_test_minmax.csv')
y_train = pd.read_csv(path+'y_train_os.csv')
y_test = pd.read_csv(path+'y_test.csv')
X_train.shape,y_train.shape

((6064, 11), (6064, 1))

In [1]:
X_train.head(1)

NameError: name 'X_train' is not defined

### 1. Random Forest Classifier

In [15]:
# Only test-wise I used a baseline model (RandomForestClassifier) to see if the pre-processing was okay (so this notebook cell is just a test):
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Instantiating the model (e.g., RandomForestClassifier) and train the model on the scaled training data (y_train does not need to be scaled since it is binary)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Create predictions
y_pred = model.predict(X_test)

# Evaluate model performance using a classification report
report = classification_report(y_test, y_pred)
print("Classification Report:")
print(report)

C:\Users\fscoh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.89      0.88       322
           1       0.95      0.95      0.95       758

    accuracy                           0.93      1080
   macro avg       0.92      0.92      0.92      1080
weighted avg       0.93      0.93      0.93      1080



### 2. SVM Classifier

In [16]:
from sklearn.svm import SVC

# Instantiate the model and train it
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train, y_train)

# Create predictions
svm_pred = svm_model.predict(X_test)

# Evaluate model performance using a classification report
svm_report = classification_report(y_test, svm_pred)
print("Classification Report:")
print(svm_report)

C:\Users\fscoh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.85      0.81       322
           1       0.93      0.89      0.91       758

    accuracy                           0.88      1080
   macro avg       0.85      0.87      0.86      1080
weighted avg       0.89      0.88      0.88      1080



### 3. XGBoost Classifier

In [19]:
from xgboost import XGBClassifier

# Instantiate the model and train it
xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train, y_train)

# Create predictions
xgb_pred = xgb_model.predict(X_test)

# Evaluate model performance using a classification report
xgb_report = classification_report(y_test, xgb_pred)
print("Classification Report:")
print(xgb_report)

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.89      0.88       322
           1       0.95      0.94      0.95       758

    accuracy                           0.93      1080
   macro avg       0.91      0.92      0.91      1080
weighted avg       0.93      0.93      0.93      1080



### 4. K-Nearest Neighbors Classifier

In [18]:
from sklearn.neighbors import KNeighborsClassifier

# Instantiate the model and train it
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

# Create predictions
knn_pred = knn_model.predict(X_test)

# Evaluate model performance using a classification report
knn_report = classification_report(y_test, knn_pred)
print("Classification Report:")
print(knn_report)

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.87      0.83       322
           1       0.94      0.90      0.92       758

    accuracy                           0.89      1080
   macro avg       0.87      0.89      0.88      1080
weighted avg       0.90      0.89      0.89      1080



C:\Users\fscoh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neighbors\_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


### GridsearchCV

In [20]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, make_scorer

# Define hyperparameter grids for each model
param_grids = {
    "RandomForest": {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5, 10]
    },
    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale", "auto"]
    },
    "XGBoost": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 0.2],
        "max_depth": [3, 5, 7]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    }
}


In [21]:
# Define models
models = {
    "RandomForest": RandomForestClassifier(random_state=42),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "KNN": KNeighborsClassifier()
}
# Use F1-score as the optimization metric
scorer = make_scorer(f1_score, average='binary')

In [22]:
best_models = {}
for name, model in models.items():
    print(f" Tuning {name} for F1-score...")
    grid_search = GridSearchCV(model, param_grids[name], cv=5, scoring=scorer, n_jobs=-1)
    grid_search.fit(X_train_ros, y_train_ros)
    
    best_models[name] = grid_search.best_estimator_
    print(f" Best {name} Params: {grid_search.best_params_}")
    print(f" Best {name} F1-score: {grid_search.best_score_:.4f}\n")

# Evaluate the best models on the test set
print("Final Model Evaluations on Test Data:")
for name, model in best_models.items():
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    print(f" {name}: F1-score = {f1:.4f}")

 Tuning RandomForest for F1-score...


C:\Users\fscoh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


 Best RandomForest Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
 Best RandomForest F1-score: 0.9573

 Tuning SVM for F1-score...


C:\Users\fscoh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


 Best SVM Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
 Best SVM F1-score: 0.8888

 Tuning XGBoost for F1-score...


C:\Users\fscoh\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:158: UserWarning: [17:36:49] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


 Best XGBoost Params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}
 Best XGBoost F1-score: 0.9560

 Tuning KNN for F1-score...


C:\Users\fscoh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neighbors\_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


 Best KNN Params: {'metric': 'manhattan', 'n_neighbors': 9, 'weights': 'distance'}
 Best KNN F1-score: 0.9414

Final Model Evaluations on Test Data:
 RandomForest: F1-score = 0.9505
 SVM: F1-score = 0.9262
 XGBoost: F1-score = 0.9465
 KNN: F1-score = 0.9341


In [23]:
print('Best model: Random Forest')
print('Best parameters: {max_depth: None, min_samples_split: 2, n_estimators: 200}')

Best model: Random Forest
Best parameters: {max_depth: None, min_samples_split: 2, n_estimators: 200}


In [51]:
model = RandomForestClassifier(max_depth= None, min_samples_split= 2, n_estimators= 200,random_state=42)
model.fit(X_train_ros, y_train_ros)

# Create predictions
y_pred = model.predict(X_test)
y_train_pred=model.predict(X_train_ros)
# Evaluate model performance using a classification report
report = classification_report(y_test, y_pred)
print("Classification Report:")
print(report)


C:\Users\fscoh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.89      0.88       322
           1       0.95      0.95      0.95       758

    accuracy                           0.93      1080
   macro avg       0.92      0.92      0.92      1080
weighted avg       0.93      0.93      0.93      1080



In [54]:
#confusion matrix
from sklearn.metrics import confusion_matrix
cm_test=confusion_matrix(y_test,y_pred)
cm_train=confusion_matrix(y_train_ros,y_train_pred)
print('confusion matrix for test set')
print(cm_test)
print('confusion matrix for train set')
print(cm_train)

confusion matrix for test set
[[285  37]
 [ 38 720]]
confusion matrix for train set
[[3032    0]
 [   0 3032]]


In [56]:
#f1 score
f1_score_test=f1_score(y_test,y_pred)
f1_score_train=f1_score(y_train_ros,y_train_pred)
print('f1_score_test =',f1_score_test)
print('f1_score_train =',f1_score_train)

f1_score_test = 0.9504950495049505
f1_score_train = 1.0


### Deep Learning tensorflow

In [67]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


model=Sequential()
model.add(Dense(units=10,activation='tanh',input_shape=(11,)))
model.add(Dense(units=8,activation='tanh'))
model.add(Dense(units=6,activation='tanh'))
model.add(Dense(units=3,activation='softmax'))

C:\Users\fscoh\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\_typing\_scalars.py:12: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  _BoolLike_co = Union[bool, np.bool]


AttributeError: module 'numpy' has no attribute 'bool'.
`np.bool` was a deprecated alias for the builtin `bool`. To avoid this error in existing code, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [64]:
model.summary()

AttributeError: 'RandomForestClassifier' object has no attribute 'summary'

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
history = model.fit(X_train_ros, y_train_ros, epochs=500, batch_size=32, validation_split=0.1)

In [ ]:
test_pred = model.predict(X_test)

In [ ]:
y_test_class = y_test
y_pred_class = np.argmax(test_pred, axis=1)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test_class, y_pred_class))
sns.heatmap(confusion_matrix(y_test_class, y_pred_class), cmap='Blues', cbar=False, annot=True)